# μP transfer — talk PoC

Read-only analysis of TopTagging transfer jobs. **No training in this notebook.**

Helpers live in ``cpen.utils.transfer_plots`` — this notebook only lists jobs and calls plots.

1. **LR scan** at one or more $(D, h)$ → pick $\eta_0^\star$ (overlay sizes by color).
2. **Width scaling** at transferred $\eta_0$: test ROC AUC and background rejection vs width / parameter count.

Add new experiment cells below with their own ``JobSpec`` lists / ``TransferStudy`` instances.


In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib
import sys

import pandas as pd

# Notebooks/ is not on PYTHONPATH by default.
REPO = Path("..").resolve()
sys.path.insert(0, str(REPO / "src"))

import cpen.utils.transfer_plots as transfer_plots
importlib.reload(transfer_plots)

from cpen.utils.transfer_plots import (
    DEFAULT_FIG_DIR,
    DEFAULT_OUT_ROOT,
    JobSpec,
    TransferStudy,
    setup_mpl_style,
)

setup_mpl_style()
FIG_DIR = DEFAULT_FIG_DIR
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_ROOT = DEFAULT_OUT_ROOT
print("transfer_plots:", transfer_plots.__file__)
print("OUT_ROOT =", OUT_ROOT)
print("FIG_DIR  =", FIG_DIR)


## Hardcoded jobs

- `lr_scan` — $\eta_0$ sweeps at one or more architectures $(L, D, h)$ (plot 1 overlays each in a different color)
- `width_scale` — transferred-$\eta_0$ runs used for the scaling slide

Parquet paths come from each job’s `RUN:` line in `slurm-<job_id>.out`.
Job notes should include depth / width / heads as relevant (`L=…, D=…, h=…`). Empty `job_id`s are skipped.


In [ ]:
# CAPEN-Llama runs
LR_SCAN_JOBS: list[JobSpec] = [
    # -- L=4, D=128
    JobSpec("12295838_0", "lr_scan", "η0=0.01,  L=4, D=128, h=8, 1 ep"),
    #JobSpec("12298444_0", "lr_scan", "η0=0.025, L=4, D=128, h=8, 1 ep"),
    JobSpec("12294820_0", "lr_scan", "η0=0.05,  L=4, D=128, h=8, 1 ep"),
    JobSpec("12295839_0", "lr_scan", "η0=0.1,   L=4, D=128, h=8, 1 ep"),
    JobSpec("12295840_0", "lr_scan", "η0=0.25,  L=4, D=128, h=8, 1 ep"),
    JobSpec("12295587_0", "lr_scan", "η0=0.5,   L=4, D=128, h=8, 1 ep"),
    JobSpec("12295852_0", "lr_scan", "η0=1,    L=4, D=128, h=8, 1 ep"),    
    # -- L=4, D=256
    JobSpec("12295865_0", "lr_scan", "η0=0.01,  L=4, D=256, h=8, 5 ep"),
    #JobSpec("12298443_0", "lr_scan", "η0=0.025, L=4, D=256, h=8, 5 ep"),
    JobSpec("12295868_0", "lr_scan", "η0=0.05,  L=4, D=256, h=8, 5 ep"),
    JobSpec("12295884_0", "lr_scan", "η0=0.1,   L=4, D=256, h=8, 5 ep"),
    JobSpec("12295894_0", "lr_scan", "η0=0.25,  L=4, D=256, h=8, 5 ep"),
    JobSpec("12295910_0", "lr_scan", "η0=0.5,   L=4, D=256, h=8, 5 ep"),
    JobSpec("12295914_0", "lr_scan", "η0=1,    L=4, D=256, h=8, 5 ep"),  
    # -- L=4, D=512, H=8, N=64, 42M params
    JobSpec("12298834_0", "lr_scan", "η0=0.01,  L=4, D=512, h=8, 5 ep"),
    #JobSpec("12298841_0", "lr_scan", "η0=0.025, L=4, D=512, h=8, 5 ep"),
    JobSpec("12298858_0", "lr_scan", "η0=0.05,  L=4, D=512, h=8, 5 ep"),
    JobSpec("12298861_0", "lr_scan", "η0=0.1,   L=4, D=512, h=8, 5 ep"),
    JobSpec("12298884_0", "lr_scan", "η0=0.25,  L=4, D=512, h=8, 5 ep"),
    JobSpec("12298899_0", "lr_scan", "η0=0.5,   L=4, D=512, h=8, 5 ep"),
    JobSpec("12298910_0", "lr_scan", "η0=1,     L=4, D=512, h=8, 5 ep"),  
    # -- L=4, D=1024, H=16, N=64, 167M params
    JobSpec("12341151_0", "lr_scan", "η0=0.01,  L=4, D=1024, h=16, 10 ep"),
    JobSpec("12341154_0", "lr_scan", "η0=0.05,  L=4, D=1024, h=16, 10 ep"),
    JobSpec("12341159_0", "lr_scan", "η0=0.1,   L=4, D=1024, h=16, 10 ep"),
    JobSpec("12341180_0", "lr_scan", "η0=0.25,  L=4, D=1024, h=16, 10 ep"),
    JobSpec("12347807_0", "lr_scan", "η0=0.5,   L=4, D=1024, h=16, 10 ep"),
    JobSpec("12349638_0", "lr_scan", "η0=1,     L=4, D=1024, h=16, 10 ep"),  
]

# μP-transferred width ladder (CLI η0 fixed; code applies /√D).
WIDTH_SCALE_JOBS: list[JobSpec] = [
    #JobSpec("12011857_0", "width_scale", "D=256, h=4, η0=0.1, 10 ep"),
    #JobSpec("12012596_0", "width_scale", "D=512, h=4, η0=0.1, 10 ep"),
    #JobSpec("12038203_0", "width_scale", "D=768, h=4, η0=0.1, 10 ep"),
]

ALL_JOBS = LR_SCAN_JOBS + WIDTH_SCALE_JOBS
D_REF = 128

for j in ALL_JOBS:
    print(f"{j.job_id:14s}  {j.role:12s}  {j.note}")


study = TransferStudy.from_jobs(ALL_JOBS, d_ref=D_REF)

display(study.summary("lr_scan"))
study.print_eta0_star("lr_scan")

print("\nWidth-scale — final test metrics:")
width_summary = study.summary("width_scale")
if width_summary.empty:
    print("(none yet)")
else:
    display(width_summary)

study.plot_lr_scan("lr_scan", stem="transfer_lr_scan_test")

In [ ]:
# CPEN RUNS
LR_SCAN_JOBS: list[JobSpec] = [
    # -- L=2, D=128 --
    JobSpec("12427729_0", "lr_scan", "η0=0.005, L=2,D=128, 20 ep"),
    JobSpec("12394615_0", "lr_scan", "η0=0.01,  L=2,D=128, 20 ep"),
    JobSpec("12394638_0", "lr_scan", "η0=0.05,  L=2,D=128, 20 ep"),
    JobSpec("12394639_0", "lr_scan", "η0=0.1,   L=2,D=128, 20 ep"),
    JobSpec("12394640_0", "lr_scan", "η0=0.25,  L=2,D=128, 20 ep"),
    JobSpec("12394659_0", "lr_scan", "η0=0.5,   L=2,D=128, 20 ep"),
    JobSpec("12394662_0", "lr_scan", "η0=1,     L=2,D=128, 20 ep"),
    JobSpec("12427715_0", "lr_scan", "η0=5,     L=2,D=128, 20 ep"),
    # -- L=3 D=192 --
    #JobSpec("", "lr_scan", "η0=0.005, D=128, h=4, 1 ep"),
    #JobSpec("", "lr_scan", "η0=0.01, L=3,D=192, 5 ep"),
    #JobSpec("", "lr_scan", "η0=0.05, L=3,D=192, 5 ep"),
    #JobSpec("", "lr_scan", "η0=0.1,  L=3,D=192, 5 ep"),
    #JobSpec("", "lr_scan", "η0=0.25, L=3,D=192, 5 ep"),
    #JobSpec("", "lr_scan", "η0=0.5,  L=3,D=192, 5 ep"),
    #JobSpec("", "lr_scan", "η0=1,    L=3,D=192, 5 ep"),
    #JobSpec("", "lr_scan", "η0=5,    L=3,D=192, 5 ep"),
    # -- L=4 D=256 --
    JobSpec("12427730_0", "lr_scan", "η0=0.005, L=4,D=256, 20 ep"),
    JobSpec("12391964_0", "lr_scan", "η0=0.01,  L=4,D=256, 20 ep"),
    JobSpec("12391968_0", "lr_scan", "η0=0.05,  L=4,D=256, 20 ep"),
    JobSpec("12391985_0", "lr_scan", "η0=0.1,   L=4,D=256, 20 ep"),
    JobSpec("12391987_0", "lr_scan", "η0=0.25,  L=4,D=256, 20 ep"),
    JobSpec("12391988_0", "lr_scan", "η0=0.5,   L=4,D=256, 20 ep"),
    JobSpec("12392006_0", "lr_scan", "η0=1,     L=4,D=256, 20 ep"),
    JobSpec("12427733_0", "lr_scan", "η0=5,     L=4,D=256, 20 ep"),
    # -- L=6 D=512
    JobSpec("12428016_0", "lr_scan", "η0=0.005, L=6,D=512, 5 ep"),
    JobSpec("12388071_0", "lr_scan", "η0=0.01, L=6,D=512, 5 ep"),
    JobSpec("12388089_0", "lr_scan", "η0=0.05, L=6,D=512, 5 ep"),
    JobSpec("12388093_0", "lr_scan", "η0=0.1,  L=6,D=512, 5 ep"),
    JobSpec("12388095_0", "lr_scan", "η0=0.25, L=6,D=512, 5 ep"),
    JobSpec("12388120_0", "lr_scan", "η0=0.5,  L=6,D=512, 5 ep"),
    JobSpec("12388121_0", "lr_scan", "η0=1,    L=6,D=512, 5 ep"),
    JobSpec("12428015_0", "lr_scan", "η0=5,    L=6,D=512, 5 ep"),
    # -- L=8 D=1024
    #JobSpec("12360511_0", "lr_scan", "η0=0.01, L=8,D=1024, 5 ep"),
    #JobSpec("", "lr_scan", "η0=0.05, L=8,D=1024, 5 ep"),
    #JobSpec("", "lr_scan", "η0=0.1, L=8,D=1024, 5 ep"),
    #JobSpec("", "lr_scan", "η0=0.25, L=8,D=1024, 5 ep"),
    #JobSpec("", "lr_scan", "η0=0.5, L=8,D=1024, 5 ep"),
    #JobSpec("", "lr_scan", "η0=1, L=8,D=1024, 5 ep"),
]

# μP-transferred width ladder (CLI η0 fixed; code applies /√D).
WIDTH_SCALE_JOBS: list[JobSpec] = [
    #JobSpec("12011857_0", "width_scale", "D=256, h=4, η0=0.1, 10 ep"),
    #JobSpec("12012596_0", "width_scale", "D=512, h=4, η0=0.1, 10 ep"),
    #JobSpec("12038203_0", "width_scale", "D=768, h=4, η0=0.1, 10 ep"),
]

ALL_JOBS = LR_SCAN_JOBS + WIDTH_SCALE_JOBS
D_REF = 128

for j in ALL_JOBS:
    print(f"{j.job_id:14s}  {j.role:12s}  {j.note}")


study = TransferStudy.from_jobs(ALL_JOBS, d_ref=D_REF)

display(study.summary("lr_scan"))
study.print_eta0_star("lr_scan")

print("\nWidth-scale — final test metrics:")
width_summary = study.summary("width_scale")
if width_summary.empty:
    print("(none yet)")
else:
    display(width_summary)

study.plot_lr_scan("lr_scan", stem="transfer_lr_scan_test")

In [ ]:
# CPEN RUNS
LR_SCAN_JOBS: list[JobSpec] = [
    # -- L=2, D=128 --
    #JobSpec("", "lr_scan", "η0=0.005, D=128, h=4, 1 ep"),
    # -- L=4 D=256 --
    JobSpec("12378613_0", "lr_scan", "η0=0.01, L=4,D=256, 5 ep"),
    JobSpec("12378665_0", "lr_scan", "η0=0.025, L=4,D=256, 5 ep"),
    JobSpec("12378688_0", "lr_scan", "η0=0.05, L=4,D=256, 5 ep"),
    JobSpec("12377921_0", "lr_scan", "η0=0.1,  L=4,D=256, 5 ep"),
    JobSpec("12377924_0", "lr_scan", "η0=0.25, L=4,D=256, 5 ep"),
    JobSpec("12377941_0", "lr_scan", "η0=0.5,  L=4,D=256, 5 ep"),
    JobSpec("12377946_0", "lr_scan", "η0=0.5,  L=4,D=256, 5 ep"),
    JobSpec("12377945_0", "lr_scan", "η0=1,    L=4,D=256, 5 ep"),
    #JobSpec("", "lr_scan", "η0=5,    L=4,D=256, 5 ep"),
    # -- L=6 D=512
    JobSpec("12378580_0", "lr_scan", "η0=0.01, L=6,D=512, 5 ep"),
    JobSpec("12378545_0", "lr_scan", "η0=0.025, L=6,D=512, 5 ep"),
    JobSpec("12378542_0", "lr_scan", "η0=0.05, L=6,D=512, 5 ep"),
    JobSpec("12378145_0", "lr_scan", "η0=0.1,  L=6,D=512, 5 ep"),
    JobSpec("12378146_0", "lr_scan", "η0=0.25, L=6,D=512, 5 ep"),
    JobSpec("12378179_0", "lr_scan", "η0=0.5,  L=6,D=512, 5 ep"),
    JobSpec("12378181_0", "lr_scan", "η0=0.75,  L=6,D=512, 5 ep"),
    JobSpec("12378197_0", "lr_scan", "η0=1,    L=6,D=512, 5 ep"),
    #JobSpec("", "lr_scan", "η0=5,    L=6,D=512, 5 ep"),
    # -- L=8 D=1024
    #JobSpec("", "lr_scan", "η0=0.01, L=8,D=1024, 5 ep"),
    #JobSpec("", "lr_scan", "η0=0.05, L=8,D=1024, 5 ep"),
    #JobSpec("", "lr_scan", "η0=0.1, L=8,D=1024, 5 ep"),
    #JobSpec("", "lr_scan", "η0=0.25, L=8,D=1024, 5 ep"),
    #JobSpec("", "lr_scan", "η0=0.5, L=8,D=1024, 5 ep"),
    #JobSpec("", "lr_scan", "η0=1, L=8,D=1024, 5 ep"),
]

# μP-transferred width ladder (CLI η0 fixed; code applies /√D).
WIDTH_SCALE_JOBS: list[JobSpec] = [
    #JobSpec("12011857_0", "width_scale", "D=256, h=4, η0=0.1, 10 ep"),
    #JobSpec("12012596_0", "width_scale", "D=512, h=4, η0=0.1, 10 ep"),
    #JobSpec("12038203_0", "width_scale", "D=768, h=4, η0=0.1, 10 ep"),
]

ALL_JOBS = LR_SCAN_JOBS + WIDTH_SCALE_JOBS
D_REF = 128

for j in ALL_JOBS:
    print(f"{j.job_id:14s}  {j.role:12s}  {j.note}")


study = TransferStudy.from_jobs(ALL_JOBS, d_ref=D_REF)

display(study.summary("lr_scan"))
study.print_eta0_star("lr_scan")

print("\nWidth-scale — final test metrics:")
width_summary = study.summary("width_scale")
if width_summary.empty:
    print("(none yet)")
else:
    display(width_summary)

study.plot_lr_scan("lr_scan", stem="transfer_lr_scan_test")

## 1. LR scan vs $\eta_0$ (all model architectures)

One color per $(L, D, h)$. Top: test ROC AUC / background rejection. Bottom: train / val loss. Dashed vlines mark each architecture’s $\eta_0^\star$.


In [ ]:
study.plot_lr_scan("lr_scan", stem="transfer_lr_scan_test")


## 2. Optimization trajectories (LR scan)

Train / val curves for a reference architecture. Default filters to ``width=D_REF``; pass ``depth=…`` / ``heads=…`` when those vary.


In [ ]:
# Add depth=… / heads=… when the LR scan mixes architectures.
study.plot_train_curves("lr_scan", width=D_REF, stem="transfer_lr_scan_curves")


## 3. Width / parameter scaling (talk slide)

Uses jobs tagged ``width_scale``. Series split by depth / heads when those vary (not only by $\eta_0$).


In [ ]:
study.plot_width_scaling("width_scale")


## Job status


In [ ]:
display(study.status())


## New experiment cell template

Copy this cell for another configuration (e.g. plain CPEN, different dropout, …).
Use a distinct ``role`` string so summaries / plots stay separated.


In [ ]:
# Example — paste job IDs as they finish:
# EXP_JOBS = [
#     JobSpec("JOBID_0", "cpen_lr_scan", "η0=0.1, D=256, 10 ep"),
# ]
# if EXP_JOBS:
#     exp = TransferStudy.from_jobs(EXP_JOBS, d_ref=256)
#     display(exp.summary())
#     exp.plot_lr_scan(role="cpen_lr_scan", stem="cpen_lr_scan_test")
#     exp.plot_train_curves(role="cpen_lr_scan", width=256, stem="cpen_lr_scan_curves")
